# Visualisation de données
Ce notebook permet la visualisation de données afin de programmer correctement le fichier preprocessing.py. 

In [49]:
# Import 
import os
import sys
import logging
import pandas as pd
import xgboost as xgb

# 1. On nettoie les anciens verrous de logging spécifiques aux notebooks
for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

# 2. On configure proprement pour que ça print TOUT dans le notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],  # <-- La magie est là, ça force l'affichage
)

# Reconstruire le chemin absolu à partir de la racine du container
path_data = os.path.join("/workspace", "data")
path_data_input = os.path.join("/workspace", "data", "input", "data_scoring_credit.csv")


## Analyse rapide

In [52]:
# Test de lecture rapide du dataset
df = pd.read_csv(path_data_input)
logging.info(f"Dimensions du dataset : {df.shape}")
df.head()

2026-07-18 10:43:41,473 - INFO - Dimensions du dataset : (1200, 12)


,branch,ncust,customer,age,ed,employ,address,income,debtinc,creddebt,othdebt,default
0,3,3017,10012,28,Bac+2,7,2,44,17.7,2.990592,4.797408,Non
1,3,3017,10017,64,Bac+5 et plus,34,17,116,14.7,5.047392,12.004608,Non
2,3,3017,10030,40,Niveau bac,20,12,61,4.8,1.042368,1.885632,Non
3,3,3017,10039,30,Niveau bac,11,3,27,34.5,1.751220,7.563780,Non
4,3,3017,10071,35,Niveau bac,2,9,38,10.9,1.462126,2.679874,Oui


### Dictionnaire des variables (Data Dictionary)

*   **`ncust`** : Numéro d'identifiant interne et court du client (à supprimer avant l'entraînement).
*   **`customer`** : Identifiant unique global du client dans le système bancaire (à supprimer avant l'entraînement).
*   **`branch`** : Code de l'agence bancaire ou de la succursale de rattachement du client.
*   **`age`** : Âge du client en années.
*   **`ed`** : Niveau d'études atteint par le client.
*   **`employ`** : Ancienneté professionnelle (nombre d'années passées chez l'employeur actuel).
*   **`address`** : Stabilité résidentielle (nombre d'années passées à l'adresse actuelle).
*   **`income`** : Revenu annuel du client (exprimé en milliers d'euros).
*   **`debtinc`** (*Debt-to-Income ratio*) : Taux d'endettement global du client (en %).
*   **`creddebt`** (*Credit Debt*) : Encours de la dette liée aux cartes de crédit et crédits conso (en milliers d'euros).
*   **`othdebt`** (*Other Debt*) : Encours des autres dettes bancaires ou privées (en milliers d'euros).
*   **`default`** (Target) : Statut de défaut de paiement du client (`Oui` = en défaut / `Non` = a remboursé).

In [22]:
# Afficher le résumé statistique des variables numériques
df.describe()

,branch,ncust,customer,age,employ,address,income,debtinc,creddebt,othdebt
count,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000,1200.000000
mean,52.196667,3478.205000,257694.200000,34.130833,6.950000,6.281667,59.960000,9.967167,1.945998,3.887205
std,28.009822,863.284183,139959.135031,13.323913,9.110525,6.143054,69.831324,6.717606,2.993344,5.506351
min,3.000000,1919.000000,10012.000000,18.000000,0.000000,0.000000,12.000000,0.000000,0.000000,0.000000
25%,20.000000,2658.000000,98136.500000,23.000000,0.000000,1.000000,27.000000,4.875000,0.409002,1.111377
50%,64.000000,3491.000000,316154.000000,31.000000,3.000000,5.000000,39.000000,8.500000,0.950550,2.220708
75%,75.000000,4358.000000,370743.500000,41.250000,10.000000,9.000000,64.000000,13.600000,2.219280,4.542486
max,91.000000,4809.000000,453777.000000,79.000000,63.000000,34.000000,1079.000000,40.700000,35.972690,63.472640


In [28]:
# Résumé des variables catégorielles
df.describe(include=["object", "category"])

,ed,default
count,1200,1200
unique,5,2
top,Bac+2,Non
freq,422,752


In [51]:
# Affiche les pourcentages des var catégorielles (ex: Bac+2  35.16)
logging.info(df['ed'].value_counts(normalize=True) * 100)

logging.info(df['default'].value_counts(normalize=True) * 100)

2026-07-18 10:43:20,027 - INFO - ed
Bac+2            35.166667
Bac+3            22.333333
Bac+4            20.666667
Niveau bac       16.000000
Bac+5 et plus     5.833333
Name: proportion, dtype: float64
2026-07-18 10:43:20,030 - INFO - default
Non    62.666667
Oui    37.333333
Name: proportion, dtype: float64


In [27]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 12 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   branch    1200 non-null   int64  
 1   ncust     1200 non-null   int64  
 2   customer  1200 non-null   int64  
 3   age       1200 non-null   int64  
 4   ed        1200 non-null   object 
 5   employ    1200 non-null   int64  
 6   address   1200 non-null   int64  
 7   income    1200 non-null   int64  
 8   debtinc   1200 non-null   float64
 9   creddebt  1200 non-null   float64
 10  othdebt   1200 non-null   float64
 11  default   1200 non-null   object 
dtypes: float64(3), int64(7), object(2)
memory usage: 112.6+ KB


## Prétraitement des données

### Nettoyage

In [57]:
logging.info(f"Nombre de valeurs manquantes : {df.isnull().sum().sum()}")  # Vérifie les valeurs manquantes dans le dataset

data_cleaned = df.dropna()  # Supprime les lignes avec des valeurs manquantes
logging.info(f"Dimensions du dataset après suppression des valeurs manquantes : {data_cleaned.shape}")

data_cleaned = data_cleaned.drop_duplicates()  # Supprime les doublons
logging.info(f"Dimensions du dataset après suppression des doublons : {data_cleaned.shape}")

data_cleaned = data_cleaned.drop(columns=['ncust', 'customer'])  # Supprime la colonne 'id' si elle existe
logging.info(f"Dimensions du dataset après suppression des colonnes 'ncust' et 'customer' : {data_cleaned.shape}")

2026-07-18 10:45:23,619 - INFO - Nombre de valeurs manquantes : 0
2026-07-18 10:45:23,621 - INFO - Dimensions du dataset après suppression des valeurs manquantes : (1200, 12)
2026-07-18 10:45:23,625 - INFO - Dimensions du dataset après suppression des doublons : (1200, 12)
2026-07-18 10:45:23,627 - INFO - Dimensions du dataset après suppression des colonnes 'ncust' et 'customer' : (1200, 10)


In [58]:
data_cleaned

,branch,age,ed,employ,address,income,debtinc,creddebt,othdebt,default
0,3,28,Bac+2,7,2,44,17.7,2.990592,4.797408,Non
1,3,64,Bac+5 et plus,34,17,116,14.7,5.047392,12.004608,Non
2,3,40,Niveau bac,20,12,61,4.8,1.042368,1.885632,Non
3,3,30,Niveau bac,11,3,27,34.5,1.751220,7.563780,Non
4,3,35,Niveau bac,2,9,38,10.9,1.462126,2.679874,Oui
...,...,...,...,...,...,...,...,...,...,...
1195,91,31,Bac+2,3,6,24,12.9,0.736848,2.359152,Oui
1196,91,37,Bac+2,10,8,43,3.6,0.806508,0.741492,Non
1197,91,25,Bac+5 et plus,0,3,16,3.2,0.288256,0.223744,Non
1198,91,34,Niveau bac,10,8,41,14.5,1.194945,4.750055,Non


### Data type

### Découpage dataset train, test, pred